In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
DATASET_PATH = "../../data/raw/"
REPORTS_DIR = "../../reports"

In [4]:
CONFIG = {
    "target_col": "Price",
    "exclude_cols": ["Image_url"]
}

In [5]:
def save_plot(fig, name):
    
    # Zapis utworzonego wykresu do folderu "reports"
    path = os.path.join(REPORTS_DIR, name)
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    print(f"Zapisano wykres: {path}")
    return name

In [6]:
def load_and_filter():
    
    # Ładowanie pliku z rozszerzeniem .csv
    csv_files = [f for f in os.listdir(DATASET_PATH) if f.endswith('.csv')]
    if not csv_files:
        print(f"Brak plików CSV w katalogu {DATASET_PATH}")
        return None
    else:
        # Ładowanie pierwszego znalezionego pliku
        file_path = os.path.join(DATASET_PATH, csv_files[0])
        print(f"Wczytywanie pliku: {file_path}")
        df = pd.read_csv(file_path)

        # Usuwanie kolumn wykluczonych w konfiguracji
        to_drop = [c for c in CONFIG["exclude_cols"] if c in df.columns]
        if to_drop:
            print(f"Wykluczanie kolumn: {to_drop}")
            df = df.drop(columns=to_drop)

        print(f"Kształt danych po filtrowaniu: {df.shape}")

    # Wyświetlenie próbki danych
    display(df.head())
    return df

In [7]:
def analyze_missing(df, suffix=""):
    
    # Analiza brakujących wartości
    missing = df.isnull().sum().to_frame(name="Missing").sort_values("Missing", ascending=False)
    missing = missing[missing["Missing"] > 0]

    print("\nTabela z liczbą brakujących wartości dla każdej kolumny")
    if missing.empty:
        print("Dataset nie zawiera brakujących wartości")
    else:
        display(missing)

    fig = plt.figure(figsize=(12, 6))
    # Heatmapa brakujących wartości
    sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
    plt.title("Mapa brakujących wartości w datasecie", fontsize=14)
    save_plot(fig, f"01_heatmapa_brakow{suffix}.png")

In [8]:
def summarize_distinct_values(df, columns):

    # Metoda wyświetlająca tabelę unikalnych wartości dla kolumn
    for col in columns:
        if col not in df.columns:
            print(f"Brak kolumny '{col}' w DataFrame.")
            continue
            
        print(f"\nPodsumowanie dla kolumny: {col}")
        
        # Zliczanie ilości oraz procentowego udziału
        counts = df[col].value_counts(dropna=False)
        percentages = df[col].value_counts(dropna=False, normalize=True) * 100
        
        # Tworzenie ramki danych dla czytelnego widoku
        summary_df = pd.DataFrame({
            'Liczba wystąpień': counts,
            'Procent (%)': percentages.round(2)
        })
        
        # Wyświetlenie tabeli
        display(summary_df)

In [9]:
def summarize_numerical_ranges(df, column, bins, labels=None):
    """
    Kategoryzuje dane numeryczne do podanych przedziałów (bins) i wyświetla tabelę.
    
    Parametry:
    - column: nazwa kolumny liczbowej (string)
    - bins: lista z granicami przedziałów (np. [0, 1000, 2000, 3000])
    - labels: opcjonalna lista etykiet dla przedziałów (np. ['<1.0L', '1.0-2.0L', '>2.0L'])
    """
    if column not in df.columns:
        print(f"Brak kolumny '{column}' w DataFrame.")
        return

    print(f"\nRozkład przedziałów dla kolumny: {column}")
    
    # Tworzenie tymczasowej serii z podzielonymi danymi
    binned_data = pd.cut(df[column], bins=bins, labels=labels)
    
    counts = binned_data.value_counts(sort=False)
    percentages = binned_data.value_counts(sort=False, normalize=True) * 100
    
    summary_df = pd.DataFrame({
        'Przedział': counts.index,
        'Liczba wystąpień': counts.values,
        'Procent (%)': percentages.values.round(2)
    }).set_index('Przedział')
    
    display(summary_df)

In [10]:
def custom_describe(df, columns):
    # Generowanie tabeli ze statystykami opisowymi (mediana, średnia, min, max, std, itp.) 
    # dla wybranych kolumn numerycznych.
    
    # Filtrowanie tylko tych kolumn, które faktycznie są w DataFrame
    valid_cols = [col for col in columns if col in df.columns]
    
    if not valid_cols:
        print("Brak podanych kolumn w DataFrame.")
        return
        
    print(f"\nStatystyki opisowe dla kolumn: {', '.join(valid_cols)}")
    
    # Wykorzystanie metody agg do obliczenia wybranych statystyk
    summary_df = df[valid_cols].agg([
        'count',           # Liczba niepustych rekordów
        'mean',            # Średnia
        'median',          # Mediana
        'min',             # Minimum
        'max',             # Maksimum
        'std'              # Odchylenie standardowe
    ]).T # Transpozycja (.T) (kolumny w wierszach)
    
    # Dodanie kwantyli dla pełnego obrazu
    summary_df['Q1 (25%)'] = df[valid_cols].quantile(0.25)
    summary_df['Q3 (75%)'] = df[valid_cols].quantile(0.75)
    
    # Formatowanie wyświetlania liczb zmiennoprzecinkowych
    display(summary_df.round(2))

In [11]:
def show_top_records(df, columns, n=10):
    """
    Wyświetla top N wierszy z najwyższymi wartościami dla wskazanych kolumn liczbowych.
    (w celu analizy wartości odstających)
    
    Parametry:
    - columns: lista kolumn do przeanalizowania
    - n: liczba rekordów do wyświetlenia (domyślnie 10)
    """
    for col in columns:
        # Sprawdzenie, czy kolumna istnieje w datasecie
        if col not in df.columns:
            print(f"Brak kolumny '{col}' w DataFrame.")
            continue
            
        # Sprawdzenie, czy kolumna jest numeryczna (aby uniknąć błędów)
        if not pd.api.types.is_numeric_dtype(df[col]):
            print(f"Kolumna '{col}' nie jest typu numerycznego.")
            continue

        # Algorytm wyświetla N najmniejszych wartości dla kolumny "Year"    
        if col.lower() == 'year':
            print(f"\nTop {n} najstarszych pojazdów: {col}")
            result_df = df.nsmallest(n, columns=col)
        else:
            print(f"\nTop {n} rekordów z najwyższą wartością w kolumnie: {col}")
            result_df = df.nlargest(n, columns=col)
        
        # Wyświetlenie wyniku w czytelnej formie
        display(result_df)

In [12]:
def analyze_correlations_before_cleaning(df):

    df_corr_before_cleaning = df.copy()

    # 1. Zamiana True/False na 1 i 0
    bool_cols = ['Full_Service_History', 'Non_Smoker_Vehicle']
    for col in bool_cols:
        if col in df_corr_before_cleaning.columns:
            df_corr_before_cleaning[col] = df_corr_before_cleaning[col].astype(int)

    # 2. Wybranie tylko kolumn liczbowych do macierzy (pomija Markę, Model, Kolor itp.)
    numerical_df = df_corr_before_cleaning.select_dtypes(include=['number'])

    # 3. Rysowanie samej mapy ciepła
    fig = plt.figure(figsize=(16, 12))
    sns.heatmap(numerical_df.corr(), cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, cbar_kws={"shrink": .8})

    plt.title('Rozszerzona macierz korelacji', fontsize=16)
    plt.tight_layout()
    save_plot(fig, "02_korelacje_przed_czyszczeniem.png")

    # Wyświetlenie 10 najsilniejszych korelacji z ceną:
    print("Najsilniejsze korelacje z kolumną Price:")
    print(numerical_df.corr()['Price'].sort_values(key=abs, ascending=False).head(10))

In [13]:
def plot_engine_and_gearbox(df):
    # 1. Filtrowanie danych do wykresów 
    # (ignorowanie możliwych błędów z danymi ograniczając liczbę biegów i cylindrów)
    df_plot = df[(df['Gears'] <= 12) & (df['Cylinders'] <= 13)].copy()

    # 2. Grupowanie Engine_Size
    df_plot['Engine_Size_Bins'] = pd.cut(df_plot['Engine_Size_cc'], 
                                         bins=[0, 1000, 2000, 3000, 4000, 10000], 
                                         labels=['<1.0L', '1.0-2.0L', '2.0-3.0L', '3.0-4.0L', '>4.0L'])

    # Wykres 1: Zależność Pojemność silnika od Liczby cylindrów (od 0 do 13)
    fig1 = plt.figure(figsize=(10, 6))
    sns.countplot(data=df_plot, x='Engine_Size_Bins', hue='Cylinders')
    plt.title('Liczba cylindrów w zależności od pojemności')
    plt.legend(title='Cylindry', loc='upper right')
    save_plot(fig1, "03a_zaleznosc_silnik_pojemnosc_vs_cylindry.png")

    # Wykres 2: Rozkład Pojemność silnika od Mocy silnika
    fig2 = plt.figure(figsize=(10, 6))
    sns.boxplot(data=df_plot, x='Engine_Size_Bins', y='Power_hp', hue='Engine_Size_Bins', palette='Blues', showfliers=False, legend=False) 
    plt.title('Moc silnika (KM) w podziale na pojemność')
    plt.xlabel('Grupa pojemności')
    plt.ylabel('Moc (HP)')
    save_plot(fig2, "03b_rozklad_moc_vs_pojemnosc.png")

    # Wykres 3: Zależność Skrzynia biegów od Liczby biegów (od 0 do 12)
    fig3 = plt.figure(figsize=(10, 6))
    sns.countplot(data=df_plot, x='Gearbox', hue='Gears')
    plt.title('Liczba biegów w typach skrzyń')
    plt.legend(title='Biegi', ncol=2)
    save_plot(fig3, "03c_zaleznosc_skrzynia_vs_biegi.png")

    # Wykres 4: Rozkład Skrzynia biegów od Liczby biegów (od 0 do 12)
    fig4 = plt.figure(figsize=(10, 5))
    sns.boxplot(data=df_plot, x='Gearbox', y='Gears', hue='Gearbox', palette='Blues', showfliers=False, legend=False)
    plt.title('Rozkład liczby biegów')
    plt.xlabel('Typ skrzyni biegów')
    plt.ylabel('Liczba biegów')
    save_plot(fig4, "03d_rozklad_skrzynia_vs_biegi.png")

    # Sprawdzenie korelacji liczbowej
    numerical_cols = ['Gears', 'Year', 'Power_hp', 'Engine_Size_cc', 'Price', 'Mileage_km']
    correlation_matrix = df_plot[numerical_cols].corr()

    # Wyświetlenie korelacji samej kolumny Gears
    print(correlation_matrix['Gears'].sort_values(ascending=False))

In [14]:
def analyze_price_distribution(df):
    # Analiza rozkładu kolumny docelowej (Price).
    # Pokazuje skośność surowych danych oraz efekt transformacji log1p.
    price = df['Price']

    skewness_raw  = price.skew()
    skewness_log  = np.log1p(price).skew()

    print(f"Statystyki Price:")
    print(f"  Skośność (surowe):  {skewness_raw:.2f}  {'silna skośność' if abs(skewness_raw) > 1 else 'ok'}")
    print(f"  Skośność (log1p):   {skewness_log:.2f}  {'silna skośność' if abs(skewness_log) > 1 else 'ok'}")

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    # Wykres 1: Oryginalny rozkład z linią mediany i średniej
    axes[0].hist(price, bins=100, color='steelblue', edgecolor='none')
    axes[0].axvline(price.median(), color='orange', linewidth=1.5, linestyle='--', label=f'Mediana: {price.median():,.0f}')
    axes[0].axvline(price.mean(),   color='red',    linewidth=1.5, linestyle='--', label=f'Średnia: {price.mean():,.0f}')
    axes[0].set_title(f'Price surowe dane\nSkośność: {skewness_raw:.2f}', fontsize=12)
    axes[0].set_xlabel('Price w Euro')
    axes[0].legend(fontsize=9)

    # Wykres 2: Po transformacji log1p
    log_price = np.log1p(price)
    axes[1].hist(log_price, bins=100, color='seagreen', edgecolor='none')
    axes[1].axvline(log_price.median(), color='orange', linewidth=1.5, linestyle='--', label=f'Mediana: {log_price.median():.2f}')
    axes[1].axvline(log_price.mean(),   color='red',    linewidth=1.5, linestyle='--', label=f'Średnia: {log_price.mean():.2f}')
    axes[1].set_title(f'Kolumna Price po log1p\nSkośność: {skewness_log:.2f}', fontsize=12)
    axes[1].set_xlabel('log1p(Price)')
    axes[1].legend(fontsize=9)

    plt.suptitle('Analiza rozkładu kolumny docelowej: Price', fontsize=14, y=1.01)
    plt.tight_layout()
    save_plot(fig, "03e_rozklad_price.png")

In [15]:
def clean_data(df):
    # Usunięcie kolumn do analizy
    df = df.drop(columns=['Fuel_Consumption_l', 'Previous_Owners', 'Gears', 'Cylinders', 'Upholstery', 'Engine_Size_cc', 'Drivetrain'])

    # Obliczanie mediany z danych koumn
    median_value_year = df['Year'].median()
    median_value_doors = df['Doors'].median()
    median_value_seats = df['Seats'].median()

    # Uzupełnienie braków medianą
    df['Year'] = df['Year'].fillna(median_value_year)
    df['Doors'] = df['Doors'].fillna(median_value_doors)
    df['Seats'] = df['Seats'].fillna(median_value_seats)

    # Uzupełnienie braków w kolumnach kategorycznych wartością "Unknown"
    df['Color'] = df['Color'].fillna("Unknown")
    df['Country'] = df['Country'].fillna("Unknown")

    df.dropna(inplace=True)

    # Wyświetlenie próbki danych
    display(df.head())
    return df

In [16]:
def analyze_outliers(df):
    print(f"Liczba wszystkich wierszy i kolumn po usunięciu kolumn z brakami: {df.shape}")

    # 1. Określenie kolumn do utworzenia wykresów boxplot w celu zobrazowania wartości odstających.
    # Brak określenia nazw kolumn spowoduje, że wszystkie kolumny numeryczne z datasetu zostaną wzięte pod uwagę.
    CHOSEN_COLUMNS = ['Price', 'Mileage_km', 'Power_hp', 'Year', 'Doors', 'Seats']

    # 2. Wybór kolumn numerycznych
    numeric_all = df.select_dtypes(include=[np.number]).columns.tolist()
    cols_to_analyze = [c for c in numeric_all if c in CHOSEN_COLUMNS]

    if not cols_to_analyze:
        print("Brak kolumn numerycznych w datasecie.")
    else:
        print(f"Analiza wartości odstających dla {len(cols_to_analyze)} kolumn.")
        
        # Przygotowanie listy do tabeli podsumowującej
        outlier_summary = []
        
        # Pętla tworząca osobny plik dla każdej kolumny
        for col in cols_to_analyze:
            # Liczenie kwantyli i przedziałów
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Zliczanie outlierów
            outliers_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
            perc = (outliers_count / len(df)) * 100
            
            outlier_summary.append({
                'Kolumna': col,
                'Liczba Wartości odstających': outliers_count,
                'Udział %': f"{perc:.2f}%",
                'Min (IQR)': lower_bound,
                'Max (IQR)': upper_bound
            })

            # Ustawianie parametrów dla tworzonych wykresów
            fig, ax = plt.subplots(figsize=(10, 5))
            sns.boxplot(x=df[col], ax=ax, color='skyblue', fliersize=4)
            ax.set_title(f'{col}\nOutliery: {outliers_count} ({perc:.1f}%)')
            ax.set_xlabel('')
            
            # Zapis każdego wykresu jako oddzielny plik
            save_plot(fig, f"04_outlier_{col}.png")

        # Wyświetlenie tabeli sumującej wartości odstające
        print("Podsumowanie wartości odstających (Metoda IQR)")
        summary_df = pd.DataFrame(outlier_summary)
        display(summary_df.sort_values(by='Liczba Wartości odstających', ascending=False))

In [17]:
def analyze_correlations(df):
    df_corr = df.copy()

    # 1. Zamiana True/False na 1 i 0
    bool_cols = ['Full_Service_History', 'Non_Smoker_Vehicle']
    for col in bool_cols:
        if col in df_corr.columns:
            df_corr[col] = df_corr[col].astype(int)

    # 2. Przypisanie dozwolonych wartości dla skrzyni biegów i rodzaju paliwa w celu
    # usunięcia rzadkich wystąpień by nie były wliczane niepotrzebnie do mapy korelacji
    allowed_gearboxes = ['Manual', 'Automatic']
    allowed_fuels = ['Gasoline', 'Diesel', 'Electric/Gasoline', 'Electric']

    # 3. Filtrowanie datasetu w celu pozostawienia najważniejszych zmiennych dla mapy korelacji
    df_filtered = df_corr[df_corr['Gearbox'].isin(allowed_gearboxes)]
    df_filtered = df_filtered[df_filtered['Fuel_Type'].isin(allowed_fuels)]

    print(f"Liczba wierszy przed filtrowaniem: {df_corr.shape[0]}")
    print(f"Liczba wierszy po filtrowaniu: {df_filtered.shape[0]}")

    # 4. One-Hot Encoding dla najważniejszych kategorii
    cat_cols = ['Gearbox', 'Fuel_Type', 'Condition', 'Seller']
    existing_cat_cols = [c for c in cat_cols if c in df_filtered.columns]

    df_filtered = pd.get_dummies(df_filtered, columns=existing_cat_cols, drop_first=True, dtype=int)

    # 5. Wybranie tylko kolumn liczbowych do macierzy (pomija Markę, Model, Kolor itp.)
    numerical_df = df_filtered.select_dtypes(include=['number'])

    # 6. Rysowanie samej mapy ciepła (heatmap)
    fig = plt.figure(figsize=(16, 12))
    sns.heatmap(numerical_df.corr(), cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, cbar_kws={"shrink": .8})

    plt.title('Rozszerzona macierz korelacji (ze zmiennymi kategorycznymi)', fontsize=16)
    plt.tight_layout()
    save_plot(fig, "05_korelacje.png")

    # Wyświetlenie 10 najsilniejszych korelacji z ceną:
    print("Najsilniejsze korelacje z kolumną Price:")
    print(numerical_df.corr()['Price'].sort_values(key=abs, ascending=False).head(10))

In [18]:
os.makedirs(REPORTS_DIR, exist_ok=True)

df = load_and_filter()
if df is not None:
    analyze_missing(df, suffix="_przed_czyszczeniem")
    summarize_distinct_values(df, ['Make', 'Body', 'Country', 'Condition', 'Fuel_Type', 'Drivetrain', 
                               'Gearbox', 'Gears', 'Cylinders', 'Seats', 'Doors', 'Color', 
                               'Upholstery', 'Seller'])
    
    summarize_numerical_ranges(df, 'Engine_Size_cc', [0, 1000, 2000, 3000, 4000, 10000], 
                               ['<1.0L', '1.0-2.0L', '2.0-3.0L', '3.0-4.0L', '>4.0L'])
    
    summarize_numerical_ranges(df, 'Power_hp', [0, 200, 400, 600, 800, 1000, 2000], 
                               ['<200', '200-400', '400-600', '600-800', '800-1000', '>1000'])
    
    summarize_numerical_ranges(df, 'Mileage_km', [0, 50000, 100000, 200000, 300000, 400000, 2000000],
                               ['<50k', '50k-100k', '100k-200k', '200k-300k', '300k-400k', '>400k'])

    custom_describe(df, ['Year', 'Doors', 'Seats'])
    
    analyze_correlations_before_cleaning(df)
    plot_engine_and_gearbox(df)
    
    analyze_price_distribution(df)
    
    df = clean_data(df)
    analyze_missing(df, suffix="_po_czyszczeniu")
    
    analyze_outliers(df)

    show_top_records(df, ['Power_hp', 'Price', 'Year', 'Doors', 'Seats'], n=10)

    show_top_records(df, ['Mileage_km'], n=20)
    
    analyze_correlations(df)

    display(df.sort_values(by='Year', ascending=True))

Wczytywanie pliku: ../../data/raw/fullGas.csv
Wykluczanie kolumn: ['Image_url']
Kształt danych po filtrowaniu: (40494, 24)


,Make,Model,Body,Mileage_km,Price,Year,Country,Condition,Fuel_Type,Fuel_Consumption_l,...,Engine_Size_cc,Cylinders,Seats,Doors,Color,Upholstery,Full_Service_History,Non_Smoker_Vehicle,Previous_Owners,Seller
0,Abarth,595,Compact,98000,16900,2020.0,IT,Used,Gasoline,6.7,...,1368.0,4.0,4.0,3.0,White,Full leather,False,False,NaN,Dealer
1,Abarth,595,Sedan,91500,12500,2017.0,IT,Used,Gasoline,6.0,...,1368.0,4.0,4.0,3.0,Grey,Full leather,False,True,4.0,Dealer
2,Abarth,595,Compact,40000,17990,2015.0,IT,Used,Gasoline,6.5,...,1368.0,4.0,4.0,3.0,Bronze,Full leather,True,True,1.0,Dealer
3,Abarth,500,Compact,133000,9300,2008.0,IT,Used,Gasoline,6.5,...,1368.0,4.0,4.0,3.0,Black,Cloth,True,True,2.0,Dealer
4,Abarth,595,Compact,61019,14990,2021.0,BE,Used,Gasoline,NaN,...,1368.0,4.0,4.0,3.0,Yellow,Part leather,True,True,1.0,Dealer



Tabela z liczbą brakujących wartości dla każdej kolumny


,Missing
Fuel_Consumption_l,27721
Previous_Owners,18223
Gears,15379
Cylinders,10162
Drivetrain,8634
Upholstery,7272
Engine_Size_cc,5930
Country,2592
Color,2575
Seats,1797


Zapisano wykres: ../../reports\01_heatmapa_brakow_przed_czyszczeniem.png

Podsumowanie dla kolumny: Make


...

,Make,Model,Body,Mileage_km,Price,Year,Country,Condition,Fuel_Type,Gearbox,Power_hp,Seats,Doors,Color,Full_Service_History,Non_Smoker_Vehicle,Seller
27881,Mitsubishi,Outlander,Off-Road/Pick-up,188500,10990,2014.0,FR,Used,Diesel,Manual,151,255.0,5.0,Blue,False,False,Dealer
11093,Dodge,Challenger,Coupe,59753,36990,2019.0,DE,Used,Gasoline,Automatic,377,44.0,2.0,Grey,False,True,Dealer
7578,Citroen,Jumpy,Van,138000,19999,2019.0,IT,Used,Diesel,Manual,150,9.0,5.0,Grey,True,True,Dealer
7659,Citroen,Spacetourer,Van,99800,24499,2017.0,DE,Used,Diesel,Automatic,179,9.0,4.0,Black,True,True,Dealer
7722,Citroen,Spacetourer,Van,10,36440,2025.0,DE,Used,Diesel,Automatic,179,9.0,5.0,White,True,False,Dealer
7766,Citroen,Spacetourer,Van,100500,19990,2019.0,BE,Used,Diesel,Manual,120,9.0,5.0,Black,True,True,Dealer
7908,Citroen,Jumper,Other,143016,3900,2002.0,BE,Used,Diesel,Manual,128,9.0,5.0,White,True,True,Dealer
7931,Citroen,Spacetourer,Van,77128,26890,2021.0,IT,Used,Diesel,Automatic,144,9.0,5.0,Red,False,False,Dealer
7955,Citroen,Spacetourer,Van,89988,26990,2018.0,DE,Used,Diesel,Manual,150,9.0,4.0,Black,True,True,Dealer
7966,Citroen,Spacetourer,Van,10,36440,2025.0,DE,Used,Diesel,Automatic,177,9.0,5.0,White,True,False,Dealer



Top 20 rekordów z najwyższą wartością w kolumnie: Mileage_km


,Make,Model,Body,Mileage_km,Price,Year,Country,Condition,Fuel_Type,Gearbox,Power_hp,Seats,Doors,Color,Full_Service_History,Non_Smoker_Vehicle,Seller
414,Abarth,595 Turismo,Other,16777215,8500,2013.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
3854,Bentley,Brooklands,Other,16777215,5500,1993.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
3881,Bentley,Continental GT,Other,16777215,19500,2004.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
3956,Bentley,Continental GTC,Other,16777215,29500,2008.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
4089,Bentley,S2,Other,16777215,22500,1960.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
7073,Chevrolet,Camaro,Other,16777215,6500,1985.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
7158,Chevrolet,Camaro,Other,16777215,4500,1997.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
20640,Lancia,Delta,Other,16777215,22500,1988.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
23382,Maserati,Quattroporte,Other,16777215,20500,2016.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer
32871,Rolls-Royce,Silver Shadow,Other,16777215,11500,1969.0,NL,Used,Gasoline,Manual,0,5.0,5.0,Black,False,False,Dealer


Liczba wierszy przed filtrowaniem: 39649
Liczba wierszy po filtrowaniu: 37793
Zapisano wykres: ../../reports\05_korelacje.png
Najsilniejsze korelacje z kolumną Price:
Price                 1.000000
Power_hp              0.473449
Doors                -0.262423
Seats                -0.162139
Gearbox_Manual       -0.144092
Condition_Used       -0.090720
Fuel_Type_Gasoline    0.088971
Mileage_km           -0.044352
Fuel_Type_Electric   -0.034544
Year                  0.024604
Name: Price, dtype: float64


,Make,Model,Body,Mileage_km,Price,Year,Country,Condition,Fuel_Type,Gearbox,Power_hp,Seats,Doors,Color,Full_Service_History,Non_Smoker_Vehicle,Seller
33312,Rolls-Royce,Phantom,Sedan,71997,63900,1929.0,NL,Used,Gasoline,Manual,120,5.0,4.0,Yellow,False,False,Dealer
33389,Rolls-Royce,Phantom,Convertible,10236,198950,1929.0,NL,Used,Gasoline,Manual,0,5.0,4.0,Blue,False,False,Dealer
33174,Rolls-Royce,Phantom,Sedan,196976,99900,1929.0,NL,Used,Gasoline,Manual,110,5.0,4.0,Red,False,False,Dealer
33335,Rolls-Royce,Phantom,Sedan,23979,119999,1935.0,DE,Used,Gasoline,Automatic,0,8.0,4.0,Black,False,False,Dealer
33247,Rolls-Royce,Phantom,Sedan,70808,150000,1936.0,FR,Used,Gasoline,Manual,170,5.0,5.0,Black,False,False,Dealer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4048,Bentley,Flying Spur,Sedan,0,350097,2026.0,NL,New,Electric/Gasoline,Automatic,782,5.0,4.0,Blue,False,False,Dealer
23709,Mazda,6,Sedan,2200,38900,2026.0,BE,Used,Electric,Automatic,258,5.0,5.0,Black,True,True,Dealer
1749,Alfa Romeo,Junior,Off-Road/Pick-up,10,28450,2026.0,IT,Used,Electric/Gasoline,Automatic,145,5.0,5.0,Grey,True,True,Dealer
12213,Ferrari,Testarossa,Coupe,1,550000,2026.0,NL,Used,Electric/Gasoline,Automatic,1050,5.0,2.0,Red,True,False,Dealer
